In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [ ]:
df_lead_paragraph = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/DateSentiment.pkl", compression=None)

In [ ]:
df_lead_paragraph

In [ ]:
# %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Sentiment/Functions.ipynb"

In [ ]:
# New words and values
new_words = {
    'crushes': 100,
    'beats': 5,
    'misses': -5,
    'trouble': -10,
    'falls': -100,
    "dividends":100,
    "dividend":100,
    "earning":100,
    "ipo":100,
    "alliance":100,
    "enhance":100,
    "enhanc":100
}
# Instantiate the sentiment intensity analyzer with the existing lexicon
vader = SentimentIntensityAnalyzer()
# Update the lexicon
vader.lexicon.update(new_words)

# 1) DATE

In [ ]:
pos_word_list=[]
neu_word_list=[]
neg_word_list=[]
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['Sentiment']] = ''
df[["lead_paragraph"]] = ''
df[["polarity"]] = ''
df[["subjectivity"]] = ''
df[["polarityB"]] = ''
df[['GI_Negative']] = ''
df[['GI_Positive']] = ''
df[['GI_Sentiment']] = ''
df[["GI_Polarity"]] = ''
df[["GI_Subjectivity"]] = ''

In [ ]:
import datetime 
import pysentiment2 as ps
hiv4 = ps.HIV4()
for date, row in df_lead_paragraph.T.iteritems():
    # try:
    sentence = unicodedata.normalize('NFKD', row['lead_paragraph'])#.encode('ascii','ignore').decode('ascii')

    # sentence = str(preprocessing_text(sentence))
    print(date)
    sentence = sentence.lower()
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date,"lead_paragraph"] = sentence
    ss = vader.polarity_scores(sentence)

    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    prc = row['NewsCount']
    d = datetime.timedelta(days = 100)
    formater = "%Y-%m-%d"

    d1 = datetime.datetime.strptime(date,formater)
    date_100 = d1 - d
    date_100=str(date_100)
    mean_prc = df_lead_paragraph.loc[date_100:date]['NewsCount'].mean()
    NewsCount_factor = (prc/mean_prc)
    
    df.at[date, 'NewsCountFactor'] = NewsCount_factor
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'SentimentOrig'] = (100*(ss['neg'] / (ss['pos'] + ss['neg'])))
            df.at[date, 'Sentiment'] = (100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(NewsCount_factor)

    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    df.at[date,"polarity"] = blob.sentiment.polarity
    df.at[date,"subjectivity"] = blob.sentiment.subjectivity
    if (blob.sentiment.polarity > 0 ):
        df.at[date,"polarityB"] = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        df.at[date,"polarityB"] = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        df.at[date,"polarityB"] = "Neutral"


    tokens = hiv4.tokenize(str(sentence))  # text can be tokenized by other ways
                                # however, dict in HIV4 is preprocessed
                                # by the default tokenizer in the library
    score = hiv4.get_score(tokens)

    df.at[date, 'GI_Negative'] = score["Negative"]
    df.at[date, 'GI_Positive'] = score["Positive"]
    df.at[date, 'GI_Polarity'] = score["Polarity"]
    df.at[date, 'GI_Subjectivity'] = score["Subjectivity"]
    if ((score['Positive'] + score['Negative']) != 0):
        number = 100*(score['Positive'] / (score['Positive'] + score['Negative']))
        if (number != ''):
            df.at[date, 'GI_SentimentOrig'] = (100*(score['Negative'] / (score['Positive'] + score['Negative'])))
            df.at[date, 'GI_Sentiment'] = ((100*(score['Negative'] / (score['Positive'] + score['Negative'])))*NewsCount_factor)

df['Sentiment'].apply(np.float)
df['GI_Sentiment'].apply(np.float)
df_lead_paragraph.drop(columns={"lead_paragraph"},inplace=True)
df = df.join(df_lead_paragraph)
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/Sentiment.pkl")

In [ ]:
# del df
# del df_lead_paragraph

In [ ]:
df#["Sentiment"].plot()

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

In [ ]:
df["Negative"].mean()

# 2) HOUR

In [ ]:
# Reading the saved data pickle file
df_lead_paragraph = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDateHourDF.pkl', compression=None)
# del df 

## Vader

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

rows_list = []
for date, row in df_lead_paragraph.T.iteritems():
    Hour = int(row["Hour"])
    print(Hour)
    sentence = unicodedata.normalize('NFKD', row['lead_paragraph'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")

    prc = row['NewsCount']
    ss = vader.polarity_scores(sentence)

    Compound = ss['compound']
    Neg = ss['neg']
    Neu = ss['neu']
    Pos = ss['pos']
    companieslist = list(filter(None, row['Company'].split(". ")))
    mcf = marketCapFactor(companieslist, date)
    MarketCapFactor = mcf
    Sentiment=0
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            Sentiment = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))*mcf

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
    
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    print(date)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    polarityB = ""
    if (blob.sentiment.polarity > 0 ):
        polarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        polarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        polarityB = "Neutral"

    new_row = {"Date":date, "Hour": Hour, "compound":Compound, "neg":Neg, "neu":Neu, "pos":Pos, "sentiment":Sentiment, "polarity": Polarity, "subjectivity":Subjectivity, "polarityB":polarityB,"lead_paragraph":sentence, "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "NewsCount":row["NewsCount"], "MarketCapFactor":MarketCapFactor}
    rows_list.append(new_row)
df = pd.DataFrame(data=rows_list, columns=["Date", "Hour", "compound", "neg", "neu", "pos", "sentiment", "polarity", "subjectivity", "polarityB","lead_paragraph","Title","Text", "Company", "NewsCount", "MarketCapFactor"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentHourly.pkl")

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

In [ ]:
del df

# 3) TRADING HOURS 09-16

In [ ]:
# Reading the saved data pickle file
df_lead_paragraph = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_09_15.pkl', compression=None)

In [ ]:
df_lead_paragraph

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_lead_paragraph.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_lead_paragraph.loc[date, 'lead_paragraph']))
    sentence = unicodedata.normalize('NFKD', row['lead_paragraph'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'lead_paragraph'] = sentence
    prc = row['NewsCount']

    ss = vader.polarity_scores(sentence)

    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']

In [ ]:
df_lead_paragraph.drop(columns={"lead_paragraph"},inplace=True)
df = df.join(df_lead_paragraph)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_09_15.pkl")

In [ ]:
del df

# 4) Not Trading Hours 16-09

In [ ]:
df_lead_paragraph = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_16_08.pkl', compression=None)

In [ ]:
df_lead_paragraph

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_lead_paragraph.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_lead_paragraph.loc[date, 'lead_paragraph']))
    sentence = unicodedata.normalize('NFKD', df_lead_paragraph.loc[date, 'lead_paragraph'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'lead_paragraph'] = sentence
    prc = row['NewsCount']

    ss = vader.polarity_scores(sentence)

    #print(sentence)
    #print("#2")
    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']

In [ ]:
df_lead_paragraph.drop(columns={"lead_paragraph"},inplace=True)
df = df.join(df_lead_paragraph)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_16_08.pkl")

In [ ]:
del df

# 5) TIME

In [ ]:
df_lead_paragraph = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl', compression=None)

In [ ]:
df_lead_paragraph.loc[df_lead_paragraph["Date"]>"2020-06-01"]

In [ ]:
# del df

# df  = pd.DataFrame()
# # Adding new columns to the data frame
# df[['compound']] = ''
# df[['neg']] = ''
# df[['neu']] = ''
# df[['pos']] = ''
# df[['sentiment']] = ''

In [ ]:
df_lead_paragraph.loc[df_lead_paragraph["Date"]>"2020-06-01"]

In [ ]:
rows_list = []
for date, row in df_lead_paragraph.T.iteritems():

    sentence = unicodedata.normalize('NFKD', row['lead_paragraph'])#.encode('ascii','ignore').decode('ascii')
    # sentence = str(preprocessing_text(sentence))

    ss = vader.polarity_scores(sentence)

    Hour = row["Hour"]
    Time = row["Time"]
    Date = row["Date"]
    Compound = ss['compound']
    Neg = ss['neg']
    Neu = ss['neu']
    Pos = ss['pos']
    Sentiment=0
    if ((Pos + Neg) != 0):
        number = 100*(Pos / (Pos + Neg))
        if (number != ''):
            SentimentScore = 100*((Neg + Neu) / (Pos + Neg + Neu))
    
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    polarityB = ""
    if (blob.sentiment.polarity > 0 ):
        polarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        polarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        polarityB = "Neutral"

    new_row = {"Time":Time, "Date":Date, "Hour": Hour, "Sentiment":Sentiment, "compound":Compound, "neg":Neg, "neu":Neu, "pos":Pos, "SentimentScore":SentimentScore, "polarity": Polarity, "subjectivity":Subjectivity, "polarityB":polarityB,"lead_paragraph":row["lead_paragraph"], "Title":row["Title"], "Text":row["Text"], "Company":row["Company"]}
    rows_list.append(new_row)

In [ ]:
df = pd.DataFrame(data=rows_list, columns=["Time", "Date", "Hour", "compound", "neg", "neu", "pos", "sentiment", "polarity", "subjectivity", "polarityB","lead_paragraph","Title","Text", "Company"])

In [ ]:
# print("Number of Negative words="+str(countNeg))
# print("Number of Positive words="+str(countPos))
# print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTime.pkl")

In [ ]:
df["sentiment"].ewm(10).mean().plot(figsize=(20,10))

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTime.pkl")

In [ ]:
df.loc[df_lead_paragraph["Date"]>"2020-06-01"]

In [ ]:
df.index = df.Time

In [ ]:
df.loc[df["Company"]=="BBY"]["polarity"].plot(figsize=(20,10))

In [ ]:
# df["sentiment"].ffill(inplace=True)